# E8: a large language model rescores and corrects the N-best lists of CTC

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ivannamor/multilingual-asr-lab/blob/main/notebooks/e8_llm_colab.ipynb)

Experiment E8 of [RESULTS.md](https://github.com/ivannamor/multilingual-asr-lab/blob/main/RESULTS.md)
gives Qwen2.5-7B-Instruct the 5 best transcripts of the CTC model with the language model of E4,
committed in `experiments/runs/e8-nbest-*`, and asks it three things: the probability of each
transcript (rescoring), which transcript is correct (choosing) and the correct transcript, without
and with examples (zero-shot and few-shot correction). The model, its revision and the prompts are
fixed in
[`experiments/E8/correction.toml`](https://github.com/ivannamor/multilingual-asr-lab/blob/main/experiments/E8/correction.toml)
and [`experiments/E8/prompts.toml`](https://github.com/ivannamor/multilingual-asr-lab/blob/main/experiments/E8/prompts.toml).

On the dev sets only the probabilities are computed, the one thing that is tuned (on a CPU, after
this notebook); on the test sets everything. No audio is needed: the notebook reads the lists.

1. On Google Colab: *Runtime → Change runtime type → A100 GPU* (or L4): the model needs about
   16 GB of GPU memory in bfloat16, more than a T4 has free. Then *Runtime → Run all* and keep the
   tab open. The first run installs the package and restarts the session: then *Run all* again.
   Colab reports this as a crash "for an unknown reason": it is expected.
2. At the end the browser downloads `e8_llm.zip` (if it does not, download it from the *Files*
   panel): the answers of the model in `experiments/E8/results/llm`, no weights.
3. In a local clone, from the repository root: `python3 -m zipfile -e e8_llm.zip .`

If the session is interrupted, run all cells again: the finished test sets are kept.

## 1. Clone and install

In [ ]:
import importlib.util
import os
import subprocess
import sys
import time
from pathlib import Path

# Work inside a clone of the repository, so that every output records the git commit and ends up
# where the experiment scripts expect it.
if Path("../pyproject.toml").exists():  # started from notebooks/ in a local clone
    os.chdir("..")
elif not Path("pyproject.toml").exists():  # on Colab
    if not Path("multilingual-asr-lab").exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/ivannamor/multilingual-asr-lab"],
            check=True,
        )
    os.chdir("multilingual-asr-lab")
if importlib.util.find_spec("asr_lab") is None:
    # The hf extra pins the version of Transformers; Colab already has PyTorch
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", ".[hf]"], check=True)
    # pip may have upgraded packages that the kernel has already imported (numpy), and mixing two
    # versions in one process fails later: restart the session, then run all the cells again
    # (this one will skip the installation)
    # Colab reports this as a crash "for an unknown reason": it is expected. The pause lets the
    # message reach the notebook before the kernel dies.
    print("Installed: the session restarts. Run all the cells again.", flush=True)
    time.sleep(2)
    os.kill(os.getpid(), 9)
print(Path.cwd())

## 2. Settings and GPU

In [ ]:
import torch

SPEC_PATH = Path("experiments/E8/correction.toml")
EXAMPLES_PATH = Path("experiments/E8/results/examples.json")
RUNS_DIR = Path("experiments/runs")
OUTPUT_DIR = Path("experiments/E8/results/llm")

if not torch.cuda.is_available():
    raise RuntimeError("No GPU: choose Runtime -> Change runtime type -> A100 GPU")
free, _ = torch.cuda.mem_get_info()
print(f"{torch.cuda.get_device_name()}: {free / 1e9:.1f} GB free")
if free < 20e9:
    raise RuntimeError("The model needs about 16 GB and room for its batches: choose an A100 or L4")

## 3. The language model

About 15 GB of weights, downloaded from Hugging Face at the revision of the spec (a few minutes).

In [ ]:
from asr_lab.correction.experiment import load_llm

llm = load_llm(SPEC_PATH, "cuda")
print(llm.name, llm.config)

## 4. The answers

The dev sets first (probabilities only, a few minutes), then the test sets (roughly 5 to 15
minutes each on an A100).

In [ ]:
from asr_lab.correction.experiment import run_llm

outputs = run_llm(SPEC_PATH, RUNS_DIR, EXAMPLES_PATH, OUTPUT_DIR, llm)

## 5. Download the results

In [ ]:
import json
import zipfile

archive = Path("e8_llm.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as zipped:
    for output in outputs:
        for file in (output, output.with_suffix(".json")):
            zipped.write(file, file.as_posix())
        meta = json.loads(output.with_suffix(".json").read_text())
        print(f"{output.name}: {meta['seconds'] / 60:.1f} min")
print(f"{archive.resolve()}: {archive.stat().st_size / 1e6:.1f} MB")

try:
    from google.colab import files

    files.download(str(archive))
except ImportError:  # not on Colab
    pass